# Phase 92: Main Pipeline with ESM2-3B (the Untested Capacity Lever)

CryptoBench proved embedding capacity was a real, addressable lever (35M ->
650M -> 3B: AUC 0.867->0.882->0.887, MCC 0.362->0.376->0.416 at 8 seeds).
The main COACH420/HOLO4K pipeline has used ESM2-35M unchanged since
Phase 10 -- this is the one major improvement lever proven elsewhere that
was never applied here. Round-3 scale-up already plateaued (more DATA
stopped helping), so this tests whether more embedding CAPACITY picks up
where scale left off. Otherwise identical to Phase52 (same round-3
topology blob, same dihedral logic) -- only the ESM2 model changes.

## Cell 0: Install dependencies

In [ ]:
!pip install -q torch-geometric==2.6.1 biopython==1.84 transformers==4.46.3

## Cell 1: Setup, load Phase 51's topology-fused pool

In [ ]:
import gzip, json, os, time, urllib.request, warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.metrics import average_precision_score, matthews_corrcoef, roc_auc_score

warnings.filterwarnings("ignore")

def locate(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    if not c: raise FileNotFoundError(f)
    return max(c, key=lambda p: p.stat().st_size)

blob = torch.load(locate("phase51_graphs.pt"), map_location="cpu", weights_only=False)
dataset = blob["dataset"]
TOPO_DIM = len(blob["topo_names"])
N_SEQ = int(dataset[0].x.size(1)) - TOPO_DIM
print(f"chains {len(dataset):,} | node dim {dataset[0].x.size(1)} "
      f"(seq {N_SEQ} + topo {TOPO_DIM}) | splits present: "
      f"{ {s: sum(1 for d in dataset if d.split==s) for s in ('train','val','test')} }")

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e: print("GPU unusable:", e)
DEVICE = torch.device("cuda" if GPU_OK else "cpu")

## Cell 2: Recompute real backbone dihedrals (identical to Phase 10/30/37/52)

In [ ]:
from Bio.PDB import PDBParser, MMCIFParser, PPBuilder

AA3={"ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY","HIS","ILE","LEU","LYS",
     "MET","PHE","PRO","SER","THR","TRP","TYR","VAL","MSE"}
RAW=Path("/tmp/pdb_raw"); RAW.mkdir(parents=True, exist_ok=True)

need = sorted({d.pdb_id for d in dataset})
def fetch(pid):
    for ext in ("pdb","cif"):
        p=RAW/f"{pid}.{ext}.gz"
        if p.exists() and p.stat().st_size>500: return pid,p
    for ext in ("pdb","cif"):
        try:
            with urllib.request.urlopen(f"https://files.rcsb.org/download/{pid}.{ext}.gz",timeout=60) as r:
                b=r.read()
            if len(b)>500:
                p=RAW/f"{pid}.{ext}.gz"; p.write_bytes(b); return pid,p
        except Exception: continue
    return pid,None

t0=time.time()
with ThreadPoolExecutor(max_workers=16) as ex: got=list(ex.map(fetch, need))
paths={p:f for p,f in got if f}
print(f"fetched {len(paths):,}/{len(need):,} in {(time.time()-t0)/60:.1f} min")

def dihedrals_for(pid, path, cid, n_expect):
    """(N,4) sin/cos phi,psi + (N,2) validity mask, or None if misaligned."""
    try:
        parser = MMCIFParser(QUIET=True) if ".cif" in path.name else PDBParser(QUIET=True, PERMISSIVE=True)
        with gzip.open(path,"rt") as fh:
            model = next(parser.get_structure(pid, fh).get_models())
        ch = model[cid]
    except Exception:
        return None
    residues=[r for r in ch if r.get_resname().strip().upper() in AA3 and "CA" in r]
    if len(residues) != n_expect:
        return None
    ppmap={}
    for pp in PPBuilder().build_peptides(ch, aa_only=False):
        for res,(phi,psi) in zip(pp, pp.get_phi_psi_list()):
            ppmap[res.id]=(phi,psi)
    phis,psis=[],[]
    for r in residues:
        phi,psi = ppmap.get(r.id, (None,None))
        phis.append(phi); psis.append(psi)
    def enc(angles):
        v=torch.tensor([a if a is not None else 0.0 for a in angles], dtype=torch.float32)
        m=torch.tensor([a is not None for a in angles], dtype=torch.float32)
        return torch.stack([torch.sin(v)*m, torch.cos(v)*m],dim=1), m.unsqueeze(1)
    pf,pm = enc(phis); sf,sm = enc(psis)
    return torch.cat([pf,sf,pm,sm],dim=1)   # (N,6): matches Phase 1 layout exactly

from joblib import Parallel, delayed
t0=time.time()
dihed = Parallel(n_jobs=max(1, os.cpu_count() or 2), verbose=1)(
    delayed(dihedrals_for)(d.pdb_id, paths[d.pdb_id], d.chain_id, int(d.num_nodes))
    if d.pdb_id in paths else None
    for d in dataset)
n_ok = sum(1 for x in dihed if x is not None)
print(f"\ndihedrals recovered for {n_ok:,}/{len(dataset):,} chains in {(time.time()-t0)/60:.1f} min")

## Cell 3: ESM2-3B embeddings (fp16, batch=1 for memory safety -- matches Phase48's CryptoBench pattern; main-pipeline sequences capped at 1000 residues, shorter than CryptoBench's 1500 cap)

In [ ]:
from transformers import AutoTokenizer, AutoModel

ESM_NAME = "facebook/esm2_t36_3B_UR50D"   # 3B params, hidden 2560 -- the
                                           # full model, matching CryptoBench's
                                           # own SOTA baseline and this
                                           # project's own best CryptoBench
                                           # result (Phase48/50/55).
tok = AutoTokenizer.from_pretrained(ESM_NAME)
esm = AutoModel.from_pretrained(ESM_NAME).half().to(DEVICE).eval()   # fp16: inference-only, no grads needed
ESM_DIM = esm.config.hidden_size
print(f"{ESM_NAME}: hidden {ESM_DIM}, device {DEVICE}, dtype {next(esm.parameters()).dtype}")

@torch.no_grad()
def embed_batch(seqs, max_len=1002):
    enc = tok(seqs, return_tensors="pt", padding=True, truncation=True, max_length=max_len)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    out = esm(**enc).last_hidden_state          # (B, L+2, H) incl BOS/EOS
    embs = []
    for i, s in enumerate(seqs):
        n = len(s)
        embs.append(out[i, 1:1+n, :].float().cpu())   # strip BOS, keep exactly n residues
    return embs

order = sorted(range(len(dataset)), key=lambda i: len(dataset[i].seq))
BATCH = 1   # conservative: 3B params, even though seqs are capped shorter than CryptoBench's
t0=time.time()
esm_feats = [None]*len(dataset)
for bi in range(0, len(order), BATCH):
    idx = order[bi:bi+BATCH]
    seqs = [dataset[i].seq for i in idx]
    embs = embed_batch(seqs)
    for i, e in zip(idx, embs):
        esm_feats[i] = e
    if bi % (BATCH*200) == 0:
        print(f"  {bi}/{len(order)}  [{(time.time()-t0)/60:.1f}m]")

n_res = sum(int(d.num_nodes) for d in dataset)
print(f"\nESM2 embeddings for {len(dataset):,} chains / {n_res:,} residues "
      f"in {(time.time()-t0)/60:.1f} min ({1000*(time.time()-t0)/n_res:.2f} ms/residue)")
del esm; torch.cuda.empty_cache()

## Cell 4: Assemble augmented features, verify alignment, save (new output filename)

In [ ]:
kept, dropped = [], 0
for i, d in enumerate(dataset):
    if dihed[i] is None or esm_feats[i] is None:
        dropped += 1; continue
    n = int(d.num_nodes)
    if dihed[i].shape[0] != n or esm_feats[i].shape[0] != n:
        dropped += 1; continue
    onehot = d.x[:, :21]                     # first 21 cols of the original block
    topo_z = d.x[:, N_SEQ:]                  # standardized topo block, unchanged
    d.x = torch.cat([onehot, dihed[i], esm_feats[i], topo_z], dim=1)
    kept.append(d)

print(f"kept {len(kept):,}/{len(dataset):,} chains (dropped {dropped}: parse/alignment failures)")
N_PRE = 21 + 6 + ESM_DIM       # everything before the topo block
print(f"new node feature dim: {kept[0].x.size(1)}  (21 onehot + 6 dihedral + "
      f"{ESM_DIM} ESM + {TOPO_DIM} topo)")
for d in kept[:50]:
    assert torch.isfinite(d.x).all(), f"{d.pdb_id}_{d.chain_id}: non-finite features"
print("finite-value spot check: OK")

for s in ("train","val","test"):
    ds = [d for d in kept if d.split == s]
    print(f"  {s:<6} {len(ds):>5} chains")

torch.save({"dataset": kept, "n_pre_topo": N_PRE, "topo_dim": TOPO_DIM,
            "esm_dim": ESM_DIM, "pos_weight": blob["pos_weight"]},
           "/kaggle/working/phase92_graphs_esm3b.pt")
print("saved -> /kaggle/working/phase92_graphs_esm3b.pt")

---

`phase92_graphs_esm3b.pt` -- same schema as `phase52_graphs.pt` except
node dim is 2621 (21+6+2560+34) instead of 541. N_PRE/TOPO_DIM are read
dynamically from the blob, so `LocalizingEGNN`/`fit`/`run_epoch` apply
unchanged. Next: rerun the Phase20/53-style (COACH420) and
Phase24/54-style (HOLO4K) clean-training + benchmark pipelines on this
blob to see whether embedding capacity closes any remaining gap to P2Rank.